# Sesión 2 — Taller práctico: qué gráfico corresponde a cada pregunta (CONECTA, proyecto A: Retención)

**Curso:** Visualización de Datos y Gerencia de la Información  
**Sesión:** 2 de 9  
**Tipo:** Notebook práctico

## Instrucciones de entrega

- Equipos de 3 personas. Entreguen este notebook **ejecutado de inicio a fin** como
  `DATAVIZ_sesion_2_proyecto_A_grupo_Y.ipynb` en el plazo acordado en clase.
- Completen las celdas `# TODO` y escriban en las celdas **Su respuesta**.
- Pueden consultar el notebook guiado, pero escriban su propio código. Pueden apoyarse en un asistente de IA para
  concentrarse en lo importante y en las decisiones; declaren al final para qué lo usaron.

## Problemática de negocio

CONECTA S.A. es un operador de telecomunicaciones en Bogotá con dos líneas, **Hogar** (internet fijo) y **Móvil**
(pospago y prepago), con datos sintéticos del curso. La Gerencia de Retención lleva al comité de diciembre una
propuesta de campaña contra las cancelaciones para el primer trimestre de 2027. La ficha de la S1 dejó el destinatario
(el comité directivo), la decisión (en qué zonas concentrar la campaña y cuánto invertir) y la pregunta: **¿dónde se
cancela más?** Le falta el indicador. Como analistas, deben:

1. Definir el indicador y ver dónde y quién cancela más.
2. Hacer un gráfico por tipo de dato: cantidades, distribuciones, tiempo, incertidumbre y relaciones.
3. Dejar en una tabla lo que llevan a la S4, donde se escribirá la recomendación al comité (el SCQA): cada gráfico de
   hoy es una pieza de esa recomendación.

Los datos todavía no se revisan (S3): trabajen con las 19 localidades con nombre oficial. El diccionario de datos
está en `datos/diccionario_conecta.md`; los segmentos (Hogar, Móvil pospago y Móvil prepago) se explican allí.

## Preparación del entorno

In [ ]:
import pathlib

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# El notebook busca datos/ e imagenes/ en su carpeta (o en la de arriba, si está en solucionario/).
BASE = pathlib.Path(".") if pathlib.Path("datos").exists() else pathlib.Path("..")
DATOS, IMAGENES = BASE / "datos", BASE / "imagenes"
IMAGENES.mkdir(exist_ok=True)
# Lo que produce el taller se guarda en salidas/, junto al notebook que se ejecuta (el del estudiante en su carpeta;
# el del solucionario, en solucionario/salidas/): así el solucionario no deja respuestas en la carpeta del estudiante.
SALIDAS = pathlib.Path("salidas")
SALIDAS.mkdir(exist_ok=True)

PALETA = {"principal": "#0F766E", "principal_claro": "#4E9F96", "marino": "#1F2A44", "secundario": "#5B6475",
          "neutro": "#8A93A3", "fondo": "#F3F5F9", "acento": "#B85C0A", "texto": "#1F2937"}
plt.rcParams.update({
    "font.family": "Calibri", "font.size": 11, "axes.edgecolor": PALETA["secundario"],
    "axes.labelcolor": PALETA["texto"], "xtick.color": PALETA["secundario"], "ytick.color": PALETA["secundario"],
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.dpi": 150,
    "savefig.bbox": "tight"})
# Con Calibri, matplotlib no dibuja bien los tamaños enteros 8 y 9 (se usan 8.5, 9.5 o más) ni el texto a 110 ppp: figure.dpi = 150.


def miles(n, redondeo=0):
    """Entero con espacio de miles; redondeo=-3 redondea a miles."""
    return f"{round(n, redondeo):,.0f}".replace(",", " ")


def decimal(x, d=1):
    return f"{x:.{d}f}".replace(".", ",")


def titulo(fig, texto, subtitulo, y=1.04):
    fig.text(0.01, y, texto, fontsize=15, fontweight="bold", color=PALETA["texto"], ha="left")
    fig.text(0.01, y - 0.055, subtitulo, fontsize=11, color=PALETA["secundario"], ha="left")


def fuente(fig, texto):
    fig.text(0.01, -0.04, texto, fontsize=8.5, color=PALETA["secundario"], ha="left")


def guardar(fig, nombre):
    """Guarda la figura en imagenes/ (la presentación usa estos archivos) y la muestra."""
    fig.savefig(IMAGENES / nombre)
    plt.show()


def limpiar_ejes(ax, x=True, y=False):
    if x:
        ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    if y:
        ax.set_yticks([])
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)

def intervalo_proporcion(exitos, n, z=1.96):
    """Proporción con su intervalo del 95 % (OpenIntro, §5.2, p. 204) y la condición de éxito-fracaso (p. 205)."""
    p = exitos / n
    se = np.sqrt(p * (1 - p) / n)
    cumple = (exitos >= 10) & ((n - exitos) >= 10)
    return pd.DataFrame({"p": p, "inferior": p - z * se, "superior": p + z * se, "n": n, "cumple": cumple})

In [ ]:
FUENTE_CONECTA = "Fuente: CONECTA S.A., datos sintéticos del curso (nov. 2025 – oct. 2026)."
h = pd.read_csv(DATOS / "conecta_clientes_mes.csv", dtype={"mes": str})
upz = gpd.read_file(DATOS / "upz.geojson")
# Segmento del cliente (ver el diccionario): Hogar, Móvil pospago o Móvil prepago.
h["segmento"] = np.where(h["linea"] == "Hogar", "Hogar", np.where(h["modalidad"] == "Prepago", "Móvil prepago", "Móvil pospago"))
o = h[h["localidad"].isin(set(upz["localidad"]))]      # filas con el nombre oficial de la localidad
variantes_localidad = h["localidad"].nunique()
print(f"{miles(len(h))} filas; {miles(len(o))} con nombre oficial de localidad")
print(f"La columna localidad tiene {variantes_localidad} valores distintos para {upz['localidad'].nunique()} localidades oficiales")

## Tarea 1 — El indicador: dónde y quién cancela más

La S1 dejó la pregunta sin indicador. Defínanlo con su fórmula y calcúlenlo para toda la base (`tasa_general`, en %).
Luego, en una figura de dos paneles con barras **desde cero, ordenadas** y con los valores (Wilke, 2019, pp. 45–58):
la tasa por localidad (`tasa_localidad`, **dónde**) y la tasa por segmento (`tasa_segmento`, **quién**).

*Consulten en la presentación:* diapositiva 7, «Razón precio/ingreso» (sección «El indicador»); diapositiva 12, «Cantidades» (sección «El directorio de gráficos»).

In [ ]:
# TODO: calculen tasa_general, tasa_localidad y tasa_segmento y hagan los dos paneles de barras.
# tasa_general = 100 * o["cancelo"].mean()     # cada fila es un cliente-mes
# tasa_localidad = (100 * o.groupby("localidad")["cancelo"].mean()).sort_values()
# tasa_segmento = (100 * h.groupby("segmento")["cancelo"].mean()).sort_values()
# fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 6.5), gridspec_kw={"width_ratios": [1.6, 1]})
# a1.barh(...); a2.barh(...)
# titulo(fig, "...", "...")     # ¿dónde y quién?
# Su código aquí:

**Escriban el indicador con su fórmula y digan qué muestran los dos paneles: ¿dónde y quién cancela más?**

*Consulten en la presentación:* diapositiva 7, «Razón precio/ingreso» (sección «El indicador»); diapositiva 12, «Cantidades» (sección «El directorio de gráficos»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 2 — Distribuciones

1. Histograma de `ingreso_mensual_cop` (lo que paga el cliente; ver el diccionario), **un color por segmento**.
2. Distribución de `antiguedad_meses` según `cancelo`: un histograma para las filas con cancelación y otro para las
   demás, cada uno en porcentaje de su grupo (los dos grupos tienen tamaños muy distintos).

*Consulten en la presentación:* diapositiva 15, «Distribuciones» (sección «El directorio de gráficos»); diapositiva 50, «Los segmentos de clientes» (sección «Incertidumbre y la primera lista»).

In [ ]:
# TODO: hagan el histograma de lo que paga el cliente por segmento y la distribución de la antigüedad según cancelo.
# fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
# for seg, color in (("Móvil prepago", ...), ("Móvil pospago", ...), ("Hogar", ...)):
#     a1.hist(h.loc[h["segmento"] == seg, "ingreso_mensual_cop"] / 1000, bins=60, alpha=0.7, label=seg, ...)
# ceros = int((h["ingreso_mensual_cop"] == 0).sum())     # ¿cuántas filas tienen 0?
# tarifa_segmento = h[h["ingreso_mensual_cop"] > 0].groupby("segmento")["ingreso_mensual_cop"].median()
# bins = range(0, 78, 6)                                   # intervalos de seis meses
# for valor, color in ((0, PALETA["neutro"]), (1, PALETA["principal"])):
#     x = h.loc[h["cancelo"] == valor, "antiguedad_meses"]
#     a2.hist(x, bins=bins, weights=np.full(len(x), 100 / len(x)), ...)   # en % de su grupo
# nuevos = 100 * h.groupby("cancelo")["antiguedad_meses"].apply(lambda s: (s < 6).mean())
# Su código aquí:

**¿Qué observan en los dos histogramas? ¿Qué preguntarían antes de usar lo que paga el cliente?**

*Consulten en la presentación:* diapositiva 15, «Distribuciones» (sección «El directorio de gráficos»); diapositiva 50, «Los segmentos de clientes» (sección «Incertidumbre y la primera lista»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 3 — Tiempo en múltiplos pequeños

**Para qué:** ver si hay localidades que cancelan más **todo el año** o solo en algunos meses.

Si se dibujan las 19 localidades como 19 líneas en un mismo gráfico, se cruzan y no se lee ninguna. La solución son
los **múltiplos pequeños**: muchos gráficos pequeños iguales, uno por localidad, puestos en una cuadrícula y con la
**misma escala**, para que el ojo compare de un panel a otro (Wilke, 2019, pp. 255 y 258–259). En matplotlib, una
cuadrícula de 2 × 3 paneles con `plt.subplots(2, 3, sharey=True)` (`sharey=True` da la misma escala vertical).

1. **La serie:** la tasa de cancelación de cada localidad en cada mes (`serie`: filas = meses; columnas =
   localidades).
2. **Qué localidades:** seis, porque caben en la cuadrícula de 2 × 3. Escojan las seis con **más clientes**
   (`cliente_id` distintos por localidad): son las que más pesan para CONECTA y las de tasa más estable.
3. **La referencia:** en cada panel, una línea gris con el **promedio simple de las 19 localidades** en cada mes
   (el promedio de las 19 tasas, sin ponderar por clientes), rotulada. Sin ella no se sabe si una localidad está alta
   o baja.
4. **El resumen:** cuenten cuántos de los 12 meses está cada localidad por encima de ese promedio (`meses_sobre`),
   pónganlo en el título de cada panel y usen el resultado en el título del gráfico.

*Consulten en la presentación:* diapositiva 22, «Múltiplos pequeños» (sección «El directorio de gráficos»).

In [ ]:
# TODO: hagan los múltiplos pequeños de la serie mensual para las seis localidades con más clientes.
# serie = 100 * o.groupby(["mes", "localidad"])["cancelo"].mean().unstack()   # filas: mes; columnas: localidad
# promedio = serie.mean(axis=1)           # promedio simple de las 19 localidades (no ponderado)
# grandes = o.groupby("localidad")["cliente_id"].nunique().nlargest(6).index
# meses_sobre = serie[grandes].gt(promedio, axis=0).sum().sort_values(ascending=False)   # meses arriba, de más a menos
# fig, ejes = plt.subplots(2, 3, figsize=(12, 6), sharey=True)   # la misma escala en todos los paneles
# for ax, loc in zip(ejes.flat, meses_sobre.index):           # paneles ordenados por meses arriba
#     ax.plot(..., serie[loc], ...); ax.plot(..., promedio, ...)
#     ax.fill_between(x, serie[loc], promedio, where=serie[loc] > promedio, interpolate=True, alpha=0.3)   # resalta los meses arriba
# Escala: la de las seis localidades y el promedio (en líneas no hace falta empezar en cero)
# titulo(fig, "...", "...", y=1.08)
# fuente(fig, FUENTE_CONECTA)
# plt.show()
# Su código aquí:

**¿Hay localidades que cancelan más todo el año o solo en algunos meses? ¿Qué gana el lector con los múltiplos pequeños frente a un solo gráfico con todas las líneas?**

*Consulten en la presentación:* diapositiva 22, «Múltiplos pequeños» (sección «El directorio de gráficos»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 4 — Incertidumbre: una muestra aleatoria

Con toda la base, la tasa de cada localidad es la real: no hay error de muestreo. Para ver qué hace un intervalo de
confianza, trabajen como si solo tuvieran una muestra:

1. Tomen una **muestra aleatoria de 30 000 filas** cliente-mes: `o.sample(30_000, random_state=2026)` (menos del
   10 % de la base). Así se cumple la condición de independencia.
2. Calculen la tasa de cada localidad en la muestra con su intervalo del 95 % (`intervalo_proporcion`, Diez et al.,
   2019, §5.2) y agreguen la **tasa real** de toda la base (`real`).
3. Grafiquen los intervalos con la tasa real marcada, cuenten cuántos la contienen (`contienen`) y marquen las
   localidades que no cumplen la condición de éxito-fracaso.

In [ ]:
# TODO: tomen la muestra, calculen los intervalos por localidad (intervalos) y compárenlos con la tasa real.
# muestra = o.sample(30_000, random_state=2026)
# conteo = muestra.groupby("localidad")["cancelo"].agg(["sum", "size"])
# intervalos = intervalo_proporcion(conteo["sum"], conteo["size"])
# intervalos["real"] = o.groupby("localidad")["cancelo"].mean()          # toda la base
# intervalos["contiene"] = (intervalos["inferior"] <= intervalos["real"]) & (intervalos["real"] <= intervalos["superior"])
# contienen = int(intervalos.loc[intervalos["cumple"], "contiene"].sum())
# fig, ax = plt.subplots(figsize=(9, 6.5))   # un intervalo por localidad y la tasa real marcada
# Su código aquí:

**¿Qué muestra la comparación entre los intervalos de la muestra y la tasa real? ¿Qué le dirían al comité sobre las localidades que no cumplen la condición?**

*Consulten en la presentación:* diapositiva 41, «Intervalo de confianza de una proporción» (sección «Incertidumbre y la primera lista»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Tarea 5 — Relaciones: ¿qué acompaña a la cancelación?

`cancelo` solo vale 0 o 1, así que una dispersión directa no sirve (dos líneas de puntos superpuestos). En una
figura de dos paneles:

1. **Localidades:** la tasa de cada localidad (`tasa_localidad`, Tarea 1) frente a su porcentaje de filas prepago
   (`prepago_localidad`), con la correlación (`r_prepago`) en el título del panel.
2. **Quejas:** la tasa según el número de quejas del mes (0, 1, 2, 3 o más), con su intervalo. Muestren también las
   quejas medias por segmento (`quejas_segmento`), para compararlas con la tasa por segmento de la Tarea 1.

*Consulten en la presentación:* diapositiva 24, «Relaciones x–y y puntos superpuestos» (sección «El directorio de gráficos»).

In [ ]:
# TODO: calculen prepago_localidad, r_prepago, tasa_quejas y quejas_segmento, y hagan los dos paneles.
# prepago_localidad = 100 * o.groupby("localidad")["modalidad"].apply(lambda s: (s == "Prepago").mean())
# r_prepago = np.corrcoef(prepago_localidad, tasa_localidad.loc[prepago_localidad.index])[0, 1]
# por_quejas = h.assign(quejas=h["quejas_mes"].clip(upper=3)).groupby("quejas")["cancelo"].agg(["sum", "size"])
# tasa_quejas = intervalo_proporcion(por_quejas["sum"], por_quejas["size"])
# quejas_segmento = h.groupby("segmento")["quejas_mes"].mean()
# fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.8))
# a1.scatter(prepago_localidad, tasa_localidad.loc[prepago_localidad.index], ...)
# a2.errorbar(...)   # puntos con su intervalo
# Su código aquí:

**Geografía (ya hecho).** Ejecuten la celda sin modificarla: es el mapa de puntos de una muestra de clientes, con los
que cancelaron resaltados. Las coordenadas raras (nulas o fuera de Bogotá) se filtran; la S3 las corrige. Construir
mapas es tema de la S5: hoy solo hay que leerlo.

In [ ]:
# Ejecute esta celda sin modificarla y observe el resultado.
muestra = h.dropna(subset=["latitud"]).query("latitud > 0").sample(8000, random_state=1)
fig, ax = plt.subplots(figsize=(6, 7))
upz.plot(ax=ax, color=PALETA["fondo"], edgecolor="white", linewidth=0.4)
ax.scatter(muestra["longitud"], muestra["latitud"], s=1, color=PALETA["neutro"])
cancelan = muestra[muestra["cancelo"] == 1]
ax.scatter(cancelan["longitud"], cancelan["latitud"], s=5, color=PALETA["principal"])
ax.set_axis_off()
titulo(fig, "Las cancelaciones aparecen por toda la ciudad", "Muestra de 8 000 filas; en verde, las que cancelaron")
fuente(fig, FUENTE_CONECTA)
plt.show()

**¿Qué acompaña a la cancelación y qué queda por verificar? ¿Qué muestra el mapa y qué no?**

*Consulten en la presentación:* diapositiva 24, «Relaciones x–y y puntos superpuestos» (sección «El directorio de gráficos»); diapositiva 33, «Geografía» (sección «El directorio de gráficos»).

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Preguntas de interpretación

**1. La columna localidad tiene 38 valores distintos y el taller usó solo las filas con alguno de los 19 nombres oficiales. ¿Qué riesgo tiene esa decisión para las tasas por localidad?**

**Su respuesta:** *(Escriba aquí su respuesta en 2–4 oraciones.)*

## Interpretación final para el negocio: lo que llevamos a la S4

En la S4 el equipo escribirá la recomendación al comité en orden SCQA (situación, complicación, pregunta y
respuesta). Redacten un párrafo de 6 a 10 oraciones, para la Gerencia de Retención, con lo que mostraron sus
gráficos: es lo que llevan a la S4. Cubran, con sus cifras:

- **la situación:** cuánto se cancela al mes y dónde (localidades);
- **quién cancela:** el segmento y la relación con el prepago de cada localidad;
- **qué acompaña a la cancelación:** las quejas, la antigüedad y el mes;
- **cuánto confiar:** qué localidades tienen pocos clientes para concluir;
- **qué queda abierto:** qué tendrían que revisar antes de llevar esto al comité. Incluyan esta pregunta: el
  prepago cancela más en todas partes, ¿es un problema que una campaña pueda corregir o el comportamiento normal del
  segmento? (se responde en la S4).

Al final, una línea con el uso de IA.

**Su respuesta:** *(Escriban aquí su párrafo y la línea del uso de IA.)*